In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# import subprocess
# import sys
# from pathlib import Path

# package_dir = Path.cwd().resolve()
# if package_dir.name == 'examples':
#     package_dir = package_dir.parent
# else:
#     repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
#     if repo_candidate.exists():
#         package_dir = repo_candidate

# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

# import importlib
# import ce_visualization_plotly
# import ce_visualization_plotly.plugin as plotly_plugin

# importlib.reload(plotly_plugin)
# plotly_plugin.register_plotly_visualization_components()

# Plotly local factual bars

The default view shows signed contribution bars. Hover cards include the calibrated contribution interval and prediction metadata.

Key options:
- `uncertainty=True` (CE-native) or `show_uncertainty=True` — draw visible uncertainty bars
- `filter_top=N` — limit to the N highest-weight features
- `sort_by` — `"abs"` (default), `"value"`, `"interval_width"`, `"label"`, `"original"`
- `show_prediction_header=False` — hide the prediction band above the bars

This is a local factual plot, not a global explanation.

In [3]:
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from calibrated_explanations import WrapCalibratedExplainer
import ce_visualization_plotly.plugin  # noqa: F401 - registers Plotly styles

## Classification

In [4]:
X_cls, y_cls = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
)
x_proper, x_temp, y_proper, y_temp = train_test_split(
    X_cls,
    y_cls,
    test_size=0.40,
    random_state=7,
    stratify=y_cls,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
    random_state=7,
    stratify=y_temp,
)

model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=7)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

factual = explainer.explain_factual(X_query[:3])

In [5]:
factual[0].plot(style="plotly.local.factual_bars", show=True);

In [6]:
# CE's uncertainty=True maps to show_uncertainty=True inside this plugin
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    uncertainty=True,
);

## Filtering and sorting

Use `filter_top` to limit to the N features with the largest contribution, and `sort_by` to control ordering.

Sorting by `"interval_width"` highlights the features whose contribution is most uncertain — useful when uncertainty bars are also enabled.

In [7]:
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    filter_top=5,
    sort_by="interval_width",
    uncertainty=True,
);

## Hiding the prediction header

Set `show_prediction_header=False` to suppress the prediction band — useful when embedding the chart in a dashboard that already shows the prediction.

In [8]:
factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    show_prediction_header=False,
);

## Regression

In [9]:
X_reg, y_reg = make_regression(
    n_samples=600,
    n_features=8,
    n_informative=6,
    noise=8.0,
    random_state=11,
)
x_proper_reg, x_temp_reg, y_proper_reg, y_temp_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.40,
    random_state=11,
)
x_cal_reg, X_query_reg, y_cal_reg, y_query_reg = train_test_split(
    x_temp_reg,
    y_temp_reg,
    test_size=0.50,
    random_state=11,
)

reg_model = RandomForestRegressor(n_estimators=80, random_state=11)
reg_explainer = WrapCalibratedExplainer(reg_model)
reg_explainer.fit(x_proper_reg, y_proper_reg)
assert reg_explainer.fitted is True

reg_explainer.calibrate(x_cal_reg, y_cal_reg)
assert reg_explainer.calibrated is True

reg_factual = reg_explainer.explain_factual(
    X_query_reg[:3],
    low_high_percentiles=(10, 90),
)

In [10]:
reg_factual[0].plot(style="plotly.local.factual_bars", show=True);

In [11]:
reg_factual[0].plot(
    style="plotly.local.factual_bars",
    show=True,
    uncertainty=True,
    filter_top=6,
);

## HTML export

In [12]:
# Passing path= to .plot() can conflict with CE's internal kwarg handling.
# Use the returned PlotRenderResult to write the figure directly instead:
result = factual[0].plot(style="plotly.local.factual_bars", show=False)
result.figure.write_html("local_factual_bars.html")